# Transform pdf files to txt


In [1]:
import re
import os

# from pdfminer.high_level import extract_text
from PyPDF2 import PdfReader

import pandas as pd

In [2]:
# show all outputs of cell, not merely of last line (i.e. default of Jupyter Notebook)
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [3]:
# obtain locations of source folder (with pdfs) and of target folder (for txt)
data_location = "../data"
case_location_pdf = "../data/CC_judgments_pdf"
case_location_txt = "../data/CC_judgments_txt"

In [4]:
print("Current working directory: ", os.getcwd())

# os.chdir(case_location_pdf)
# print("Current working directory: ", os.getcwd())

Current working directory:  /workspaces/keywords_decisions_CoS/notebooks


## Extract text from pdf

First, let's obtain an overview of all the cases for which pdf files were obtained and which need to be transformed to txt files.

In [5]:
# # Set working directory to location folder (since os.walk works on working directory)
# os.chdir(case_location_pdf)

# obtain list of all cases in current folder
cases_in_folder = []
for dir_name, dirs, files in os.walk("../data/CC_judgments_pdf"):
    cases_in_folder.extend(files) # use extend instead of append to avoid nestd list

In [6]:
# assess results
len(cases_in_folder)
cases_in_folder[:10]
type(cases_in_folder)

61

['2025-001n.pdf',
 '2025-002n.pdf',
 '2025-003n.pdf',
 '2025-004n.pdf',
 '2025-005n.pdf',
 '2025-006n.pdf',
 '2025-007n.pdf',
 '2025-008n.pdf',
 '2025-009n.pdf',
 '2025-010n.pdf']

list

Then we define a function to read the texts of the pdf files, disregarding the heades and footers.

In [7]:
def extract_text_body_only(pdf_path, y_margin=50):
    """
    Extracts text from a PDF while excluding header and footer content
    using pypdf's visiting_body approach.

    Args:
        pdf_path (str): Path to the PDF file.
        y_margin (int): Number of points to treat as header/footer margin from top and bottom.

    Returns:
        str: The extracted body text.
    """
    reader = PdfReader(pdf_path)
    output = []

    for page in reader.pages:
        page_height = float(page.mediabox.top)

        # print("Page height: ", page_height)

        body_lines = []

        def visitor_body(text, cm, tm, fontDict, fontSize):
            nonlocal body_lines
            y = tm[5]  # Y position from transformation matrix
            if y_margin < y < (page_height - y_margin):
                body_lines.append((y, text))

        page.extract_text(visitor_text=visitor_body)

        # Optional: sort lines by Y descending (PDFs often render top-down)
        # body_lines.sort(reverse=True)
        
        text_only = [line for y, line in body_lines]
        
        # Append text to output list (without additional newlines between lines)
        # output.append("\n".join(text_only))
        output.append("".join(text_only))

    # return "\n\n".join(output)
    full_text = "".join(output)

    # Count amount of words in full text
    word_count = len(re.findall(r'\w+', full_text))
    
    # Return the full text and its length
    return full_text, word_count  


In [8]:
temp_text, temp_count = extract_text_body_only("../data/CC_judgments_pdf/2025-001n.pdf")
print("Word count: ", temp_count)
print("Text: ", temp_text)

Word count:  5215
Text:  ECLI:BE:GHCC:2025:ARR.001  
Grondwettelijk Hof  
 
 
Arrest nr.  1/2025  
van 9 januari  2025  
Rolnummer  : 8115  
 
 
 In zake  : de prejudiciële vraag over artikel  1, §§ 1 en 2, van de wet van 19 juli 1991 
« betreffende de bevolkingsregisters, de identiteitskaarten, de vreemdelingenkaarten en de 
verblijfsdocumenten », gesteld door het Arbeidshof te Brussel.  
 
  Het Grondwettelijk Hof,  
  samengesteld uit de voorzitters Pierre  Nihoul en Luc  Lavrysen, en de rechters Thierry  Giet, 
Joséphine  Moerman, Michel  Pâques, Yasmine  Kherbache, Danny  Pieters, S abine  de Bethune, 
Emmanuelle  Bribosia, Willem Verrijdt, Kattrin  Jadin en Magali  Plovie, bijgesta an door griffier 
Nicolas  Dupont , onder voorzitterschap van voorzitter Pierre  Nihoul,  
  wijst na beraad het volgende arrest  : 
    I.  Onderwerp van de prejudicië le vraag en rechtspleging 
  Bij arrest van 27  november 2023, waarvan de expeditie ter griffie van het Hof is ingekomen 
op 4 decembe

Then, for each of the retrieved pdf files, we transform them to txt files. Encoding 'utf-8' is added as option to ensure that more characters can be encoded. Initially, f.e.'"' (i.e. character '\u201f') could not be read when merely applying default.

Visual inspection of the original pdf files of the cases, shows that the fonts used generally apply ANSI encoding, and occasionally Identity-h encoding. Since Python's encoding is dependent on the interaction with the filesystem and thus depends on the environment (https://stackoverflow.com/questions/42070668/python-3-default-encoding-cp1252), it is advisable to explicitly choose an encoding (i.e. UTF-8 since this allows to decode ANSI).

In [9]:
lengths_texts = []

for index, case in enumerate(cases_in_folder):
    # Check if the file is a PDF
    if case.endswith(".pdf"):
        # Construct the full path to the PDF file
        pdf_path = os.path.join(case_location_pdf, case)
        pdf_path = os.path.abspath(pdf_path)
        
        # Extract text from the PDF
        text, length_text = extract_text_body_only(pdf_path)

        # append case name (without ".pdf" i.e. final 4 charcters) and length of text to list
        lengths_texts.append((case[:-4], length_text))
        
        # Construct the full path for the output text file
        txt_path = os.path.join(case_location_txt, case.replace(".pdf", ".txt"))
        
        # Write the extracted text to a text file
        with open(txt_path, "w", encoding="utf-8") as txt_file:
            txt_file.write(text)

        # print(f"Extracted text from {case} and saved to {txt_path}")
        if index % 10 == 0: 
            print ("{} cases processed".format(index + 1))

33872

1 cases processed


18376

111920

29229

...

46695

21882

12612

4842

61 cases processed


In [10]:
# Inspect lenght of cases
length_text_df = pd.DataFrame(lengths_texts, columns=["case_number", "word_count"])
# Inspect the first few rows of the DataFrame
length_text_df.head()

,case_number,word_count
0,2025-001n,5215
1,2025-002n,2795
2,2025-003n,16487
3,2025-004n,4489
4,2025-005n,34395


In [11]:
# Store dataframe in csv and in pickle format
length_text_df.to_csv(os.path.join(data_location, "lengths_texts.csv"), index=False)
length_text_df.to_pickle(os.path.join(data_location, "lengths_texts.pkl"))

## Test chunks

In [12]:
# parts = []

# def visitor_body(text, cm, tm, font_dict, font_size):
#     y = cm[5]
#     if 50 < y < 720:
#         parts.append(text)

#  # Create function to extract text from pdf, ignoring headers and footers
# def extract_text_from_pdf(pdf_path):
#     # Read the PDF file
#     reader = PdfReader(pdf_path)

#     # Extract text of entire document
#     pdf_text = reader.extract_text()

#     # Check if the PDF has any pages
#     if len(reader.pages) == 0:
#         print(f"No pages found in {pdf_path}")
#         return ""
#     # Extract text from the first page
#     # pdf_text = extract_text(pdf_path, page_numbers=[0], maxpages=1, caching=True, laparams=LAParams(), codec='utf-8', strip_control=True)
#     page = reader.pages[0]

#     # Initialize an empty string to store the extracted text
#     pdf_text = extract_text(pdf_path, page_numbers=[0], maxpages=1, caching=True, laparams=LAParams(), codec='utf-8', strip_control=True)


#     # Create a PDF text extraction object
#     # pdf_text = extract_text(pdf_path, page_numbers=[0], maxpages=1, caching=True, laparams=LAParams(), codec='utf-8', strip_control=True)
#     # Initialize an empty string to store the extracted text
#     # pdf_text = extract_text(pdf_path, page_numbers=[0], maxpages=1, caching=True, laparams=LAParams(), codec='utf-8', strip_control=True)
#     # pdf_text = extract_text(pdf_path, page_numbers=[0], maxpages=1, caching=True, laparams=LAParams(), codec='utf-8', strip_control=True)
#     # pdf_text = extract_text(pdf_path, page_numbers=[0], maxpages=1, caching=True, laparams=LAParams(), codec='utf-8', strip_control=True)
#     text = ""

#     parts = []
    
#     # Extract text from each page
#     for page in reader.pages:
#         text += page.extract_text() + "\n"
    

    
#     return text.strip()

In [13]:
# # Set working directory to target folder to write all txt files in
# os.chdir(case_location_txt)

# for index, case_number in enumerate(cases_in_folder):
#     # Load specific pdf file
#     text = extract_text("../pdf/" + case_number)

#     # write assessed .pdf to .txt. For naming use case_number but remomve '.pdf'
#     with open(case_number[:-4] + '.txt', 'w', encoding = 'utf-8') as file:
#         file.write(text)
        
#     if index % 10 == 0: 
#         print ("{} cases processed".format(index + 1))

In [14]:
# #import module to check default encoding system (https://phrase.com/blog/posts/beginners-guide-to-locale-in-python/)
# import locale

In [15]:
# # assess default encoding 
# locale.getpreferredencoding()

In [16]:
# for index, case_number in enumerate(cases_in_folder[:10]):
#     print(index, case_number)

In [17]:
# import pdfminer.high_level
# pip install pdfminer
# pip uninstall pdfminer
# pip install pdfminer.six
# import pdfminer.six

In [18]:
# os.getcwd()

In [19]:
# # inspect extracted text
# text[:15]

In [20]:
# cases_in_folder = []
# for dir_name, dirs, files in os.walk("."):
#     cases_in_folder.append(files)
# cases_in_folder

In [21]:
# test = cases_in_folder[0]
# test[:-4]

In [22]:
# # Load specific pdf file
# text = extract_text('250270.pdf')

# # print first characters (simple print all does not work)
# print(text[0:1000])

# # write assessed .pdf to .txt
# with open('250270.txt', 'w') as file:
#     file.write(text)